**Pipeline complet :**
1. Installation des dépendances
2. Configuration
3. Chargement WorldCover 
4. Échantillonnage 10 000 points (CDL confiance ≥ 95% + WorldCover cropland)
5. Extraction Sentinel-2 → shape `(N, 36, 10)`
6. Masque données manquantes → shape `(N, 36)`
7. Normalisation ÷10000
8. Labels CDL + regroupement <5% en Others
9. **Courbes NDVI — Fig. 2 du papier**
10. Vérification dimensions + sauvegarde `.npy`

**Sources :**
- `data/raw/sentinel2/{region}/` 
- `data/raw/cdl/` 
- ESA WorldCover 2021 


In [ ]:
# Installation des dépendances
!pip install numpy rasterio pyproj tqdm boto3 s3fs matplotlib -q
print("✓ Dépendances installées")

## 1. Imports & Configuration

In [ ]:
import os
import json
import warnings
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.merge import merge
from pathlib import Path
from tqdm.notebook import tqdm
import pyproj
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from calendar import monthrange

warnings.filterwarnings("ignore")

# WorldCover depuis AWS S3 public — pas de credentials nécessaires
os.environ["AWS_NO_SIGN_REQUEST"] = "YES"

# ── BANDES DU PAPIER (Section 2.2.3) ─────────────────────────────────────────
# Band 1, 9, 10 exclus (résolution 60m — contribuent moins à la classification)
# Gardés : 3 visible, 1 NIR, 4 red-edge, 2 SWIR
BANDS = ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12"]
N_BANDS      = 10
N_TIMESTEPS  = 36
N_SAMPLES    = 10_000
CDL_CONF_MIN = 95    # papier : "95% confidence to filter CDL"
WC_CROPLAND  = 40   # ESA WorldCover classe 40 = Cropland
SEED         = 42
OUTPUT_DIR   = "data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Bandes : {BANDS}")
print(f"Shape finale : ({N_SAMPLES}, {N_TIMESTEPS}, {N_BANDS})")

## 2. Configuration régions + tuiles WorldCover AWS S3

In [ ]:
# Tuiles ESA WorldCover 2021 sur AWS S3 (bucket public ESA)
# Format : N{lat}W{lon} — coin bas-gauche de la tuile 3°×3°
WC_TILES = {
    "arkansas": [
        "s3://esa-worldcover/v200/2021/map/ESA_WorldCover_10m_2021_v200_N33W093_Map.tif",
        "s3://esa-worldcover/v200/2021/map/ESA_WorldCover_10m_2021_v200_N33W090_Map.tif",
        "s3://esa-worldcover/v200/2021/map/ESA_WorldCover_10m_2021_v200_N36W093_Map.tif",
        "s3://esa-worldcover/v200/2021/map/ESA_WorldCover_10m_2021_v200_N36W090_Map.tif",
    ],
    "california": [
        "s3://esa-worldcover/v200/2021/map/ESA_WorldCover_10m_2021_v200_N36W123_Map.tif",
        "s3://esa-worldcover/v200/2021/map/ESA_WorldCover_10m_2021_v200_N36W120_Map.tif",
        "s3://esa-worldcover/v200/2021/map/ESA_WorldCover_10m_2021_v200_N39W123_Map.tif",
        "s3://esa-worldcover/v200/2021/map/ESA_WorldCover_10m_2021_v200_N39W120_Map.tif",
    ],
}

# Configuration des régions — classes CDL du papier (Table 2)
REGIONS = {
    "arkansas": {
        "bbox":     [-92.0, 34.0, -90.0, 35.5],
        "s2_dir":   "data/raw/sentinel2/arkansas",
        "cdl_path": "data/raw/cdl/2021_30m_cdls_arkansas.tif",
        "classes":  {1: "Corn", 2: "Cotton", 3: "Rice", 5: "Soybean"},
        "n_classes": 5,   # Corn, Cotton, Rice, Soybean, Others
    },
    "california": {
        "bbox":     [-121.5, 36.5, -119.5, 38.0],
        "s2_dir":   "data/raw/sentinel2/california",
        "cdl_path": "data/raw/cdl/2021_30m_cdls_california.tif",
        "classes":  {3: "Rice", 36: "Alfalfa", 69: "Grapes", 74: "Pistachios", 75: "Almonds"},
        "n_classes": 6,   # Grapes, Rice, Alfalfa, Almonds, Pistachios, Others
    },
}
print("✓ Configuration chargée")

## 3. Fonctions — Chargement S2 + vérification des 10 bandes

In [ ]:
def load_s2_tifs(s2_dir, region_name):
    """
    Charge les 36 GeoTIFFs Sentinel-2 (output code copine).
    Vérifie que les bandes correspondent aux 10 bandes du papier :
    B02, B03, B04, B05, B06, B07, B08, B8A, B11, B12
    (Band 1, 9, 10 exclus — résolution 60m, papier Section 2.2.3)
    """
    tif_files = sorted(Path(s2_dir).glob(f"S2_{region_name}_2021_T*.tif"))
    if len(tif_files) == 0:
        raise FileNotFoundError(
            f"Aucun GeoTIFF S2 dans {s2_dir}\n"
            f"Vérifie que le téléchargement de ta copine est terminé."
        )
    print(f"  {len(tif_files)} GeoTIFFs S2 trouvés")

    # vérifie et mappe les bandes du premier fichier
    with rasterio.open(tif_files[0]) as src:
        descriptions = [src.descriptions[i] or f"band_{i+1}" for i in range(src.count)]
        print(f"  Bandes dans les GeoTIFFs : {src.count}")
        print(f"  Descriptions : {descriptions}")

        # construit le mapping : nom_bande → index_dans_tif
        band_index_map = {}
        for i, desc in enumerate(descriptions):
            for b in BANDS:
                if b in str(desc):
                    band_index_map[b] = i
                    break

        if len(band_index_map) == N_BANDS:
            print(f"  ✓ Les 10 bandes du papier trouvées et mappées correctement")
        else:
            print(f"  [!] Mapping incomplet ({len(band_index_map)}/{N_BANDS}) → fallback ordre naturel")
            band_index_map = {b: i for i, b in enumerate(BANDS[:src.count])}

    return tif_files, band_index_map

print("✓ Fonction load_s2_tifs définie")

## 4. Fonction — WorldCover en ligne (AWS S3)

In [ ]:
def load_worldcover_online(region_name, bbox):
    """
    Charge ESA WorldCover 2021 directement depuis AWS S3 (bucket public ESA).
    Lit uniquement la fenêtre de la bbox — pas de téléchargement complet.
    Papier : "ESA WorldCover 2021 to mask non-cropland areas"
    """
    west, south, east, north = bbox
    tiles = WC_TILES[region_name]

    print(f"  Chargement WorldCover depuis AWS S3 ({len(tiles)} tuiles)...")
    datasets = []
    for url in tiles:
        try:
            src = rasterio.open(url)
            b = src.bounds
            if b.right > west and b.left < east and b.top > south and b.bottom < north:
                datasets.append(src)
                print(f"    ✓ {url.split('/')[-1]}")
            else:
                src.close()
        except Exception as e:
            print(f"    ✗ {url.split('/')[-1]} — {e}")

    if not datasets:
        print("  [!] WorldCover inaccessible → filtre cropland désactivé")
        return None, None, None

    wc_merged, wc_transform = merge(datasets, bounds=(west, south, east, north))
    wc_crs = datasets[0].crs
    for ds in datasets:
        ds.close()

    wc_data = wc_merged[0]
    print(f"  WorldCover chargé : {wc_data.shape} pixels")
    return wc_data, wc_transform, wc_crs

print("✓ Fonction load_worldcover_online définie")

## 5. Fonction — Échantillonnage 10 000 points

In [ ]:
def sample_valid_points(bbox, cdl_path, region_name, n_samples, seed=42):
    """
    Tire n_samples points random dans la bbox.
    Filtre : CDL confiance ≥ 95% ET WorldCover = cropland (40)
    Papier : "randomly sampled 10,000 points in each study area"
             "set a 95% confidence to filter the CDL map"
             "ESA WorldCover 2021 to mask non-cropland areas"
    """
    np.random.seed(seed)
    west, south, east, north = bbox

    # ── CDL local ────────────────────────────────────────────
    print(f"  Ouverture CDL : {cdl_path}")
    with rasterio.open(cdl_path) as src:
        cdl_crs, cdl_transform = src.crs, src.transform
        cdl_data = src.read(1)
        cdl_conf = src.read(2) if src.count >= 2 else None
        if cdl_conf is None:
            conf_path = cdl_path.replace(".tif", "_confidence.tif")
            if os.path.exists(conf_path):
                with rasterio.open(conf_path) as c:
                    cdl_conf = c.read(1)
            else:
                print("  [!] Pas de bande confiance CDL → filtre désactivé")

    # ── WorldCover depuis AWS S3 ──────────────────────────────
    wc_data, wc_transform, wc_crs = load_worldcover_online(region_name, bbox)

    to_cdl = pyproj.Transformer.from_crs("EPSG:4326", cdl_crs, always_xy=True).transform
    to_wc  = pyproj.Transformer.from_crs("EPSG:4326", wc_crs,  always_xy=True).transform if wc_data is not None else None

    valid_points, valid_labels = [], []
    attempts, max_att = 0, n_samples * 200

    print(f"  Échantillonnage {n_samples} points agricoles...")
    with tqdm(total=n_samples, desc="  Points") as pbar:
        while len(valid_points) < n_samples and attempts < max_att:
            attempts += 1
            lon = np.random.uniform(west, east)
            lat = np.random.uniform(south, north)

            # CDL
            xc, yc = to_cdl(lon, lat)
            col_c = int((xc - cdl_transform.c) / cdl_transform.a)
            row_c = int((yc - cdl_transform.f) / cdl_transform.e)
            if not (0 <= row_c < cdl_data.shape[0] and 0 <= col_c < cdl_data.shape[1]):
                continue
            val = int(cdl_data[row_c, col_c])
            if val == 0:
                continue
            if cdl_conf is not None and int(cdl_conf[row_c, col_c]) < CDL_CONF_MIN:
                continue

            # WorldCover
            if wc_data is not None:
                xw, yw = to_wc(lon, lat)
                col_w = int((xw - wc_transform.c) / wc_transform.a)
                row_w = int((yw - wc_transform.f) / wc_transform.e)
                if not (0 <= row_w < wc_data.shape[0] and 0 <= col_w < wc_data.shape[1]):
                    continue
                if int(wc_data[row_w, col_w]) != WC_CROPLAND:
                    continue

            valid_points.append((lon, lat))
            valid_labels.append(val)
            pbar.update(1)

    if len(valid_points) < n_samples:
        print(f"  [!] Seulement {len(valid_points)} points valides — bbox peut-être trop petite")

    return np.array(valid_points, dtype=np.float32), np.array(valid_labels, dtype=np.int32)

print("✓ Fonction sample_valid_points définie")

## 6. Fonction — Extraction S2 → (N, 36, 10)

In [ ]:
def extract_s2_values(tif_files, band_index_map, points):
    """
    Extrait les valeurs S2 aux points GPS pour chaque timestep.
    Garantit l'ordre exact des 10 bandes du papier (Section 2.2.3) :
    B02, B03, B04, B05, B06, B07, B08, B8A, B11, B12

    Retourne :
        X    : (N, 36, 10) float32  — réflectances brutes (×10000)
        mask : (N, 36)     float32  — 1=valide, 0=missing
    """
    N = len(points)
    X    = np.zeros((N, N_TIMESTEPS, N_BANDS), dtype=np.float32)
    mask = np.zeros((N, N_TIMESTEPS),          dtype=np.float32)

    print(f"  Extraction : {N} points × {len(tif_files)} timesteps")
    print(f"  Ordre bandes garanti : {BANDS}")

    for tif_path in tqdm(tif_files, desc="  Timesteps S2"):
        t_idx = int(tif_path.stem.split("_T")[-1]) - 1  # 0-indexed

        with rasterio.open(tif_path) as src:
            tif_data  = src.read()   # (n_bands, H, W)
            tif_tr    = src.transform
            to_tif    = pyproj.Transformer.from_crs(
                "EPSG:4326", src.crs, always_xy=True).transform

            for i, (lon, lat) in enumerate(points):
                xt, yt = to_tif(float(lon), float(lat))
                col = int((xt - tif_tr.c) / tif_tr.a)
                row = int((yt - tif_tr.f) / tif_tr.e)

                if not (0 <= row < tif_data.shape[1] and 0 <= col < tif_data.shape[2]):
                    continue  # hors image → reste 0 (missing)

                # extrait dans l'ordre exact du papier
                for b_out, bname in enumerate(BANDS):
                    if bname in band_index_map:
                        b_in = band_index_map[bname]
                        if b_in < tif_data.shape[0]:
                            X[i, t_idx, b_out] = tif_data[b_in, row, col]

                if X[i, t_idx, :].sum() != 0:
                    mask[i, t_idx] = 1.0

    return X, mask

print("✓ Fonction extract_s2_values définie")

## 7. Fonctions — Labels CDL + Normalisation

In [ ]:
def assign_labels(raw_labels, classes_map, n_classes):
    """
    Réindexe les labels CDL.
    Classes < 5% → regroupées en "Others" (dernière classe).
    Papier : "crop types < 5% merged into Others"
    """
    others = n_classes - 1
    return np.array(
        [classes_map.get(int(l), others) for l in raw_labels],
        dtype=np.int64
    )

def normalize_s2(X):
    """
    Normalise les réflectances S2 L2A : divise par 10000.
    Papier : Sentinel-2 Level-2A surface reflectance (bottom-of-atmosphere)
    Plage après normalisation : [0, 1]
    """
    return np.clip(X / 10000.0, 0.0, 1.0)

print("✓ Fonctions assign_labels et normalize_s2 définies")

## 8. Fonction — Courbes NDVI (Fig. 2 du papier)

Reproduit exactement la **Fig. 2** de Wang et al. (2024) :
> *"NDVI time-series curve of each crop is generated by averaging the NDVI values of all corresponding crop samples"*

In [ ]:
def plot_ndvi_curves(X, y, mask, classes_map, n_classes, region_name):
    """
    Génère les courbes NDVI moyennes par culture — Fig. 2 du papier.
    Axe X : Day of Year (DOY)
    Axe Y : Mean NDVI (pixels valides uniquement, mask=1)
    """
    RED_IDX, NIR_IDX = 2, 6  # B04, B08 dans l'ordre BANDS

    # DOY du centre de chaque fenêtre de 10 jours
    doy, day = [], 1
    for month in range(1, 13):
        _, last = monthrange(2021, month)
        doy += [day+4, day+14, day+20+(last-21)//2]
        day += last
    doy = doy[:N_TIMESTEPS]

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#2196F3','#F44336','#4CAF50','#FF9800','#9C27B0','#00BCD4','#795548','#607D8B']
    class_names = list(classes_map.values()) + ["Others"]

    for cls_idx in range(n_classes):
        sel = (y == cls_idx)
        if sel.sum() == 0:
            continue
        X_c, m_c = X[sel], mask[sel]
        ndvi_t = []
        for t in range(N_TIMESTEPS):
            v = m_c[:, t].astype(bool)
            if v.sum() == 0:
                ndvi_t.append(np.nan)
                continue
            r, n = X_c[v, t, RED_IDX], X_c[v, t, NIR_IDX]
            ndvi_t.append(float(np.mean((n - r) / (n + r + 1e-8))))

        doy_v  = [doy[t] for t in range(N_TIMESTEPS) if not np.isnan(ndvi_t[t])]
        ndvi_v = [ndvi_t[t] for t in range(N_TIMESTEPS) if not np.isnan(ndvi_t[t])]

        ax.plot(doy_v, ndvi_v, label=class_names[cls_idx],
                color=colors[cls_idx % len(colors)], marker='o', markersize=4, linewidth=1.5)

    ax.set_xlabel("Day of Year", fontsize=12)
    ax.set_ylabel("Mean NDVI Value", fontsize=12)
    ax.set_title(f"NDVI Time-Series Profiles — {region_name.capitalize()}\n"
                 f"(Fig. 2 — Wang et al. 2024)", fontsize=12)
    ax.set_xlim(0, 365)
    ax.set_ylim(0, 1)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(50))
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    out = f"{OUTPUT_DIR}/{region_name}_ndvi_curves.png"
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  ✓ Courbes NDVI sauvegardées : {out}")

print("✓ Fonction plot_ndvi_curves définie")

## 9. Fonction — Comparaison CDL local vs S2 téléchargé

In [ ]:
def compare_cdl_vs_s2(points, raw_labels, X, mask, region_name, classes_map, n_classes):
    """
    Compare distribution CDL vs données S2 :
    - Distribution des classes (avec alerte si <5%)
    - Taux de données manquantes par timestep
    - NDVI moyen par classe (détecte désalignements spatiaux)
    """
    others = n_classes - 1
    N      = len(points)
    stats  = {"region": region_name, "n_samples": N, "alerts": []}
    y      = assign_labels(raw_labels, classes_map, n_classes)
    unique, counts = np.unique(y, return_counts=True)
    names  = list(classes_map.values()) + ["Others"]

    # ── distribution CDL ─────────────────────────────────────
    print(f"\n  Distribution CDL ({region_name}) :")
    cdl_dist = {}
    for cls, cnt in zip(unique, counts):
        pct  = cnt / N * 100
        name = names[cls] if cls < others else "Others"
        cdl_dist[str(cls)] = {"name": name, "count": int(cnt), "pct": round(pct, 2)}
        tag  = " ⚠️ <5%" if pct < 5 else ""
        print(f"    {name:15s} : {cnt:5d} ({pct:.1f}%){tag}")
        if pct < 5:
            stats["alerts"].append(f"{name} < 5% → sera regroupé en Others")
    stats["cdl_distribution"] = cdl_dist

    # ── données manquantes ───────────────────────────────────
    miss = (1 - mask.mean(axis=0)) * 100
    print(f"\n  Données manquantes : global={miss.mean():.1f}%  "
          f"min={miss.min():.1f}%(T{miss.argmin()+1:02d})  "
          f"max={miss.max():.1f}%(T{miss.argmax()+1:02d})")
    if miss.mean() > 50:
        msg = f"Taux manquant {miss.mean():.1f}% > 50% — vérifie les GeoTIFFs"
        stats["alerts"].append(msg)
        print(f"  ⚠️  {msg}")
    stats["missing_rate_global"]       = float(miss.mean())
    stats["missing_rate_per_timestep"] = miss.tolist()

    # ── NDVI moyen par classe ─────────────────────────────────
    RED_IDX, NIR_IDX = 2, 6
    print(f"\n  NDVI moyen par classe (pixels valides) :")
    spectral = {}
    for cls in unique:
        sel  = (y == cls)
        Xc   = X[sel]; mc = mask[sel].astype(bool)
        r, n = Xc[:,:,RED_IDX], Xc[:,:,NIR_IDX]
        ndvi = np.where(mc & ((n+r)>0), (n-r)/(n+r+1e-8), np.nan)
        mn   = float(np.nanmean(ndvi))
        name = names[cls] if cls < others else "Others"
        print(f"    {name:15s} : {mn:.3f}")
        spectral[name] = {"mean_ndvi": round(mn, 4)}
        if name != "Others" and mn < 0.05:
            msg = f"{name} NDVI très bas ({mn:.3f}) — possible désalignement CDL/S2"
            stats["alerts"].append(msg)
            print(f"    ⚠️  {msg}")
    stats["spectral_stats"] = spectral

    return stats, y

print("✓ Fonction compare_cdl_vs_s2 définie")

## 10. Pipeline principal — process_region

In [ ]:
def process_region(region_name, config):
    print(f"\n{'='*65}")
    print(f"  RÉGION : {region_name.upper()}")
    print(f"{'='*65}")

    # 1. GeoTIFFs S2 + vérification 10 bandes du papier
    tif_files, band_index_map = load_s2_tifs(config["s2_dir"], region_name)

    # 2. 10 000 points random (CDL ≥95% + WorldCover cropland)
    points, raw_labels = sample_valid_points(
        config["bbox"], config["cdl_path"], region_name, N_SAMPLES, SEED)
    print(f"\n  Points valides : {len(points)}")

    # 3. Extraction S2 → (N, 36, 10) avec ordre bandes garanti
    X, mask = extract_s2_values(tif_files, band_index_map, points)

    # 4. Normalisation ÷10000 (réflectances surface L2A)
    print("\n  Normalisation ÷10000...")
    X = normalize_s2(X)
    print(f"  X : min={X.min():.4f}  max={X.max():.4f}  mean={X.mean():.4f}")

    # 5. Labels CDL + comparaison CDL vs S2
    stats, y = compare_cdl_vs_s2(
        points, raw_labels, X, mask, region_name, config["classes"], config["n_classes"])

    # 6. Courbes NDVI — Fig. 2 du papier
    print("\n  Génération courbes NDVI (Fig. 2 du papier)...")
    plot_ndvi_curves(X, y, mask, config["classes"], config["n_classes"], region_name)

    # 7. Vérification dimensions finales
    print(f"\n  Dimensions finales :")
    print(f"    X     : {X.shape}   ← (N, 36, 10) ✓" if X.shape == (len(points), N_TIMESTEPS, N_BANDS) else f"    X     : {X.shape} ← ERREUR")
    print(f"    mask  : {mask.shape}   ← (N, 36) ✓"   if mask.shape == (len(points), N_TIMESTEPS) else f"    mask  : {mask.shape} ← ERREUR")
    print(f"    y     : {y.shape}        ← (N,) ✓"    if y.shape == (len(points),) else f"    y     : {y.shape} ← ERREUR")
    assert X.shape == (len(points), N_TIMESTEPS, N_BANDS)
    assert mask.shape == (len(points), N_TIMESTEPS)
    assert y.shape == (len(points),)
    print("  ✓ Toutes les dimensions sont correctes")

    # 8. Sauvegarde .npy
    pfx = f"{OUTPUT_DIR}/{region_name}"
    np.save(f"{pfx}_X.npy",      X)
    np.save(f"{pfx}_mask.npy",   mask)
    np.save(f"{pfx}_y.npy",      y)
    np.save(f"{pfx}_points.npy", points)
    with open(f"{pfx}_stats.json", "w") as f:
        json.dump(stats, f, indent=2)

    print(f"\n  ✓ Sauvegardé dans {OUTPUT_DIR}/ :")
    print(f"    {region_name}_X.npy       {X.shape}  → Input1 MCTNet")
    print(f"    {region_name}_mask.npy    {mask.shape}    → Input2 MCTNet")
    print(f"    {region_name}_y.npy       {y.shape}       → labels")
    print(f"    {region_name}_ndvi_curves.png          → Fig. 2 du papier")

    if stats["alerts"]:
        print(f"\n  ⚠️  {len(stats['alerts'])} alerte(s) :")
        for a in stats["alerts"]: print(f"    - {a}")
    else:
        print("  ✓ Aucune alerte — données cohérentes")

    return X, mask, y, points, stats

print("✓ Fonction process_region définie")

## 11. Exécution — Arkansas

In [ ]:
ark_X, ark_mask, ark_y, ark_points, ark_stats = process_region("arkansas", REGIONS["arkansas"])

## 12. Exécution — California

In [ ]:
cal_X, cal_mask, cal_y, cal_points, cal_stats = process_region("california", REGIONS["california"])

## 13. Rapport global + vérification finale

In [ ]:
# Sauvegarde rapport global
all_stats = {"arkansas": ark_stats, "california": cal_stats}
with open(f"{OUTPUT_DIR}/preprocessing_report.json", "w") as f:
    json.dump(all_stats, f, indent=2)

# Résumé final
print("=" * 65)
print("  PREPROCESSING TERMINÉ — Résumé")
print("=" * 65)
for name, X_, mask_, y_ in [
    ("Arkansas",   ark_X,  ark_mask,  ark_y),
    ("California", cal_X,  cal_mask,  cal_y),
]:
    miss = (1 - mask_.mean()) * 100
    print(f"\n  {name} :")
    print(f"    X shape    : {X_.shape}  (N, 36, 10)")
    print(f"    mask shape : {mask_.shape}  (N, 36)")
    print(f"    y shape    : {y_.shape}  (N,)")
    print(f"    Missing    : {miss:.1f}%")
    print(f"    Classes    : {dict(zip(*np.unique(y_, return_counts=True)))}")

print(f"\n  Fichiers dans {OUTPUT_DIR}/ :")
for f in sorted(Path(OUTPUT_DIR).glob("*")):
    sz = f.stat().st_size / 1e6
    print(f"    {f.name:<40} {sz:.1f} MB")